In [ ]:
import os
import numpy as np
import nibabel as nib
import h5py
import matplotlib.pyplot as plt

In [ ]:
# Parent Folder
dst = "Synapse Abdomen Preprocessed"
os.makedirs(dst, exist_ok=True)

# Children Folders
os.makedirs(os.path.join(dst, "Train"), exist_ok=True)
os.makedirs(os.path.join(dst, "Test"), exist_ok=True)

train_list = []
test_list = []

train_npz_dir = os.path.join(dst, "Train")
test_h5_dir  = os.path.join(dst, "Test")

In [ ]:
def save_case_h5(vol_img, vol_label, case_id, test_dir):
    fname = os.path.join(test_dir, f"case{case_id}.npy.h5")

    with h5py.File(fname, "w") as f:
        f.create_dataset("image", data=vol_img.astype(np.float32))
        f.create_dataset("label", data=vol_label.astype(np.uint8))

def save_slices_npz(vol_img, vol_label, case_id, train_dir, train_list):
    H, W, D = vol_img.shape

    for sl in range(D):
        img2d = vol_img[:, :, sl]
        lab2d = vol_label[:, :, sl]

        slice_name = f"case{case_id}_slice{(sl+1):03d}"
        fname = os.path.join(train_dir, slice_name)

        np.savez(fname,
                image = img2d.astype(np.float32),
                label = lab2d.astype(np.uint8))

        train_list.append(slice_name)

def process_case(vol_img, vol_label, case_id,
                 train_npz_dir,
                 test_h5_dir,
                 train_list,
                 test_list,
                 is_test):

    if is_test:
        save_case_h5(vol_img, vol_label, case_id, test_h5_dir)
        test_list.append(f"case{case_id}")
    else:
        save_slices_npz(vol_img, vol_label, case_id, train_npz_dir, train_list)

In [ ]:
train_list = []
test_list = []

train_npz_dir = os.path.join(dst, "Train")
test_h5_dir  = os.path.join(dst, "Test")

In [ ]:
raw_dir = '/kaggle/input/datasets/kangaonkaggle/synapse-abdomen-dataset/Abdomen'

all_lst = [
    "0031", "0007", "0009", "0005", "0026", "0039", "0024", "0034",
    "0033", "0030", "0023", "0040", "0010", "0021", "0006", "0027",
    "0028", "0037", "0008", "0022", "0038", "0036", "0032", "0002",
    "0029", "0003", "0001", "0004", "0025", "0035"
]
test = ["0008", "0022", "0038", "0036", "0032", "0002", "0029", "0003", "0001", "0004", "0025", "0035"]

# Loop through files inside Abdomen Folder
for folder_name in sorted(os.listdir(raw_dir)):
    path = os.path.join(raw_dir, folder_name)

    if folder_name == "RawData":
        # Loop through RawData
        for images in sorted(os.listdir(path)):
            if images == "Training":
                training_dir = os.path.join(path, images)
                
                img_path_nii = os.path.join(training_dir, sorted(os.listdir(training_dir))[0])
                label_path_nii = os.path.join(training_dir, sorted(os.listdir(training_dir))[1])
                print(f"Traning Image Path: {img_path_nii}\nTraining Label Path: {label_path_nii}")
                print(f"-"*60)

                for idx, nii_file in enumerate(sorted(os.listdir(img_path_nii))):
                    img_folder_id = nii_file[3:-4]
                    if img_folder_id in all_lst:
                        print(f"Processing File at Index {idx+1}. Name/Label: {nii_file}/{sorted(os.listdir(label_path_nii))[idx]}")
                        img_path_folder = os.path.join(img_path_nii, nii_file)
                        label_path_folder = os.path.join(label_path_nii, sorted(os.listdir(label_path_nii))[idx])
                        img_list = sorted(os.listdir(img_path_folder))
                        label_list = sorted(os.listdir(label_path_folder))
                        print(f"Path to Folder containing Image: {img_path_folder}\nPath to Folder containing Label: {label_path_folder}")
                        assert len(img_list) == len(label_list), f"Number of Images doesn't match Number of Labels: {len(img_list):{len(label_list)}}"
                        try:
                            for i, img in enumerate(img_list):
                                img_path = os.path.join(img_path_folder, img)
                                label_path = os.path.join(label_path_folder, label_list[i])
                                print(f"Image Path: {img_path}\nLabel Path: {label_path}")
                                load_img = nib.load(img_path)
                                load_label = nib.load(label_path)
    
                                # Clip images within [-125, 275]
                                low, high = -125, 275
                                vol_img = load_img.get_fdata().astype(np.float32)
                                vol_label = load_label.get_fdata().astype(np.int16)
                                vol_img = np.clip(vol_img, low, high)
                                assert vol_img.shape == vol_label.shape, f"Image and Label Shape Mismatch: {vol_img.shape} | {vol_label.shape}"
                                print(f"Image Shape: {vol_img.shape}\nLabel Shape: {vol_label.shape}")
    
                                # Normalize each 3D image to [0, 1]
                                vol_img = (vol_img - low) / (high - low)

                                # Save and Slice
                                process_case(
                                    vol_img,
                                    vol_label,
                                    img_folder_id,
                                    train_npz_dir,
                                    test_h5_dir,
                                    train_list,
                                    test_list,
                                    is_test = img_folder_id in test
                                    )
                                print(f"="*60)
                        
                        except Exception as err:
                            print(f"!"*60)
                            print(f"Error: {err}")
                            print(f"!"*60)
                    else:
                        print(f"="*60)
                        print(f"Skipping Folder ID: {img_folder_id}")
                        print(f"="*60)
                        continue
                        
            else:
                continue
    else:
        continue

In [ ]:
list_dir = os.path.join(dst, "Synapse Lists")
os.makedirs(list_dir, exist_ok=True)

with open(os.path.join(list_dir, "train.txt"), "w") as f:
    for name in train_list:
        f.write(name + "\n")

with open(os.path.join(list_dir, "test_vol.txt"), "w") as f:
    for name in test_list:
        f.write(name + "\n")